<a href="https://colab.research.google.com/github/azic0917/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
print('Menu Detector!')

Menu Detector!


In [13]:
# ---------------------
# Import Libraries
# ---------------------
from google.colab import drive
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np

In [14]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
# Define Dataset Path

DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print('Dataset_path:', DATASET_PATH)

CUSTOM_CLASS_MAPPING = {
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    'chocolate_cake': 'dessert', # label grouping | class consolidation
    'cheesecake': 'dessert',     # label grouping | class consolidation
    'kebab': 'kebab',
    'pilaf': 'pilaf',
    'pizza': 'pizza'
}

CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf', 'pizza']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

print('Class to Index:', CLASS_TO_IDX)
print('Number of Classes:', NUM_CLASSES)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

Dataset_path: /content/drive/MyDrive/food101_dataset
Class to Index: {'hamburger': 0, 'hot_dog': 1, 'dessert': 2, 'kebab': 3, 'pilaf': 4, 'pizza': 5}
Number of Classes: 6


In [16]:
# ---------------------
# Custom Dataset Class
# ---------------------

class FoodDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        # print('images_length:', len(self.images))
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        # print('image_path:', img_path)
        label = self.labels[idx]
        # print('label:', label)
        try:
            image = Image.open(img_path)
            if image.mode == "P" or image.mode == "RGBA":
                image = image.convert("RGBA").convert("RGB")
            else:
                image = image.convert("RGB")
        except (UnidentifiedImageError, OSError):
            print(f"Skipping broken image: {img_path}")
            return self.__getitem__((idx + 1) % len(self.images))
        if self.transform:
            image = self.transform(image)
        return image, label

In [17]:
# ---------------------
# Gather and Split Data
# ---------------------
all_images = []
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
    class_path = os.path.join(DATASET_PATH, original_class) # /content/drive/MyDrive/food101_dataset/hamburger
    print('class_path:', class_path)
    if not os.path.exists(class_path):
        print(f"Warning: {class_path} not found")
        continue
    for img in os.listdir(class_path):
        if img.endswith(('.jpg', '.jpeg', '.png')): # filter out other formats
            full_path = os.path.join(class_path, img) # /content/drive/MyDrive/food101_dataset/hamburger/100057.jpg
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images)
split = int(0.8 * len(all_images))
train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

# print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))
img, lbl = dataset[0]

class_path: /content/drive/MyDrive/food101_dataset/hamburger
class_path: /content/drive/MyDrive/food101_dataset/hot_dog
class_path: /content/drive/MyDrive/food101_dataset/chocolate_cake
class_path: /content/drive/MyDrive/food101_dataset/cheesecake
class_path: /content/drive/MyDrive/food101_dataset/kebab
class_path: /content/drive/MyDrive/food101_dataset/pilaf
class_path: /content/drive/MyDrive/food101_dataset/pizza
4149


In [18]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform)
val_dataset = FoodDataset(val_images, val_labels, transform=transform)

In [19]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2) # thread | parallel loading for speed
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [20]:
# pretrained model
model = mobilenet_v2(weights="IMAGENET1K_V1") # pretrained model | lightweight | CNN | 100 class | million+ images
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES) # fine-tuning | backbone | model layer freeze

In [21]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
model = model.to(device)

device: cuda


In [22]:
criterion = nn.CrossEntropyLoss() # Loss Function | '70%' burger, '30%' pilaf
optimizer = optim.Adam(model.parameters(), lr=0.001) # weight
torch.backends.cudnn.benchmark = True # Benchmark Setting | Trick | 10% - 20%

In [24]:
# ---------------------
# Training Loop
# ---------------------

NUM_EPOCHS = 15
best_accuracy = 0.0

for epoch in range(NUM_EPOCHS):
    model.train() # train mode
    running_loss = 0.0
    for images, labels in train_loader: # Forward and Backward(Back propagation)
      images, labels = images.to(device), labels.to(device)
      optimizer.zero_grad() # gradient => 0
      outputs = model(images) # Forward Pass | Dessert | 6 Classes
      loss = criterion(outputs, labels) # Calculate Loss
      loss.backward()
      optimizer.step() # Adam optimizer
      running_loss += loss.item() # Track Loss

  # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total # Calculate Validation Accuracy
    print(f'Epoch [{epoch + 1}/{NUM_EPOCHS}], Loss: {running_loss / len(train_loader):.4f}, Val Accuracy: {val_acc:.2f}%')

    if val_acc > best_accuracy:
        best_accuracy = val_acc
        torch.save(model.state_dict(), '/content/menu_detector.pth')
        print('New best model saved!')

Epoch [1/15], Loss: 0.6070, Val Accuracy: 84.01%
New best model saved!
Epoch [2/15], Loss: 0.3680, Val Accuracy: 90.37%
New best model saved!
Epoch [3/15], Loss: 0.3173, Val Accuracy: 84.20%
Epoch [4/15], Loss: 0.2447, Val Accuracy: 88.34%
Epoch [5/15], Loss: 0.2014, Val Accuracy: 89.21%
Epoch [6/15], Loss: 0.1689, Val Accuracy: 90.56%
New best model saved!
Epoch [7/15], Loss: 0.1675, Val Accuracy: 88.54%
Epoch [8/15], Loss: 0.1768, Val Accuracy: 87.48%
Epoch [9/15], Loss: 0.1445, Val Accuracy: 89.69%
Epoch [10/15], Loss: 0.1193, Val Accuracy: 89.02%
Epoch [11/15], Loss: 0.1328, Val Accuracy: 87.57%
Epoch [12/15], Loss: 0.0843, Val Accuracy: 90.37%
Epoch [13/15], Loss: 0.0884, Val Accuracy: 89.31%
Epoch [14/15], Loss: 0.0892, Val Accuracy: 89.11%
Epoch [15/15], Loss: 0.1159, Val Accuracy: 87.57%
